# Inspect greedy generate

Load a checkpoint, run `HMERModel.generate` on a small batch, and show each
image next to its generated sequence and its ground-truth target.

The target is only printed for comparison -- it never reaches the model.

Needs a checkpoint written by the current `train.py`: the stored config is
replayed field by field, so a checkpoint from before the config changed
will not load.

In [ ]:
# Re-import hmer_latex modules on every cell execution, so editing e.g.
# models/model.py and re-running the generate cell picks up the change.
# Methods are patched on existing instances too, so `model` stays valid --
# but a change to __init__ or to a module-level constant needs the
# model-construction cell re-run.
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from hmer_latex.data.loaders import make_loader
from hmer_latex.data.vocab import EOS, PAD, SOS
from hmer_latex.eval import load_for_inference

# Run from the repo root: the config stored in a checkpoint holds
# repo-relative paths (vocab_path, the split dirs), so they only resolve
# from there. Guarded on the directory name, so re-running is harmless.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

CHECKPOINT = Path("checkpoints/densenet_small/best.pt")
DATA_DIR = Path("data/processed/crohme_tiny")
N_SAMPLES = 8
DEVICE = torch.device("cpu")  # tiny batch; cpu keeps this reproducible and dependency-free

In [ ]:
# load_for_inference rebuilds the model from the config stored alongside the
# weights, so there is nothing here to keep in sync with train.py.
model, vocab, config = load_for_inference(CHECKPOINT, DEVICE)
pad_id, sos_id, eos_id = vocab.stoi[PAD], vocab.stoi[SOS], vocab.stoi[EOS]

ckpt = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
print(f"checkpoint epoch={ckpt['epoch']}  best_exprate={ckpt['best_exprate']:.4f}")
print(f"model={config.model}")
print(f"loaded {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M params")

In [ ]:
# First N_SAMPLES in file order -- same batch every run, so changes in the
# output are changes in generate(), not a different draw. One batch, so
# batch_size is forced to N_SAMPLES rather than whatever the run used.
config.override("data.batch_size", N_SAMPLES)
dl = make_loader(DATA_DIR, vocab, config, limit=N_SAMPLES)

imgs, img_pad_mask, tgt_in, tgt_out, tgt_mask = next(iter(dl))
imgs, img_pad_mask = imgs.to(DEVICE), img_pad_mask.to(DEVICE)
# Images are padded to the batch maximum; img_pad_mask is what keeps that
# padding out of cross-attention.
print(imgs.shape, img_pad_mask.shape, tgt_out.shape)

In [ ]:
def to_latex(ids):
    """Token ids -> string, cut at the first <eos>.

    generate() keeps decoding a row after it has emitted <eos> (every row
    runs until the whole batch is done), so the tail past the first <eos>
    is junk and has to go before comparing. Same cut eval.py makes.
    """
    ids = list(ids)
    if eos_id in ids:
        ids = ids[: ids.index(eos_id)]
    return vocab.decode(ids)


pred_ids = model.generate(imgs, img_pad_mask, sos_id=sos_id, eos_id=eos_id)
print(type(pred_ids), getattr(pred_ids, "shape", None))

preds = [to_latex(row.tolist()) for row in pred_ids.cpu()]
refs = [to_latex(row.tolist()) for row in tgt_out]

for p, r in zip(preds, refs):
    print(f"{'OK ' if p == r else '   '} pred: {p}\n     ref:  {r}")

In [ ]:
n = len(preds)
fig, axes = plt.subplots(n, 1, figsize=(10, 2.6 * n), squeeze=False)
for ax, img, pred, ref in zip(axes[:, 0], imgs.cpu(), preds, refs):
    # the transform inverted polarity (ink=1), so gray_r shows black ink on white
    ax.imshow(img.squeeze(0), cmap="gray_r")
    ax.set_axis_off()
    match = "==" if pred == ref else "!="
    ax.set_title(f"pred: {pred}\nref:  {ref}   [{match}]", fontsize=9, loc="left", family="monospace")
fig.tight_layout()